ACOLITE Example 1: MSI processing

This notebook allows you to run ACOLITE processing on Google Colab. The required dependencies are installed and the latest version is cloned from GitHub (https://github.com/acolite/acolite).

In this example, Sentinel-2 scenes are retrieved from Google Cloud Storage/Earth Engine buckets, processed and stored in a temporary cloud environment. If you set up your credentials, scenes from CDSE and EarthExplorer can also be downloaded using the API in ACOLITE. Ancillary data can be retrieved if EarthData credentials are set up, but are not essential for this example.

Author: Quinten Vanhellemont, RBINS, quinten.vanhellemont@naturalsciences.be

Written 2026-05-11 for VUB Oceans and Lakes course

In [ ]:
# general imports
import os, sys

# install dependencies for acolite
# for colab: should be done in another manner when running locally
# for colab: assume if netCDF4 is installed the other dependencies are as well
try:
  import netCDF4
except:
  !pip install numpy matplotlib scipy gdal pyproj scikit-image pyhdf pyresample netcdf4 h5py requests pygrib cartopy

In [ ]:
# clone acolite repo if it does not exist
if not os.path.exists('acolite'):
  !git clone --depth 1 https://github.com/acolite/acolite

In [ ]:
# add acolite to path and import
sys.path.append('acolite')
import acolite as ac

In [ ]:
## if authorised, use Google Cloud Service data
use_gcs = True

# auth is needed for retrieving S2 data from GCS
# this will ask you to give permissions to this notebook to access your Google Drive
if use_gcs:
  from google.colab import auth
  auth.authenticate_user()

## otherwise we will use the CDSE API for download
## and credentials need to be imported
else:
  # load credentials and set as env variables
  credentials = ac.shared.import_config('credentials.txt')
  for k in credentials: os.environ[k] = credentials[k]

In [ ]:
## location and date of interest
## you can use the Copernicus browser to find images
## https://browser.dataspace.copernicus.eu/
region_name = 'Oostende'
station_lon = 2.90
station_lat = 51.25
date = '2026-05-09'
date = '2026-05-01'

In [ ]:
## create point ROI for query
roi = 'POINT ({} {})'.format(station_lon, station_lat)

In [ ]:
## use ACOLITE API to query CDSE
## CDSE credentials not needed for query
urls, scenes = ac.api.cdse.query(roi = roi, collection = "SENTINEL-2", start_date = date, end_date = date)
## print the results
for scene in scenes:
  print(scene)

In [ ]:
## make input directory
if not os.path.exists('Input/'):
  os.makedirs('Input/')

## get copy of the L1C bundles
local_scenes = []
for si, scene in enumerate(scenes):
  local_scene = 'Input/{}'.format(scene)
  if os.path.exists(local_scene):
    print('We have {}'.format(local_scene))
    local_scenes.append(local_scene)
    continue

  ## use GCS to retrieve the S2 scene
  if use_gcs:
    ## figure out scene location on GCS
    if scene.startswith('S2'):
      tile = scene.split('_')[-2]
      zone = tile[1:3]
      t1 = tile[3]
      t2 = tile[4:6]
      gs_url = 'gs://gcp-public-data-sentinel-2/tiles/{}/{}/{}/{}'.format(zone, t1, t2, scene)
      print('Scene url: {}'.format(gs_url))
    else:
      print('{} not supported'.format(scene))

    # let's get the scene from GCS
    if not os.path.exists(local_scene):
      !gsutil -m cp -r $gs_url Input/

  ## use ACOLITE API to download the scenes from CDSE
  ## CDSE needed for download
  if not os.path.exists(local_scene):
    local_scene = ac.api.cdse.download(urls[si], scenes = scenes[si], output = 'Input/')[0]

  ## add to local scenes list
  if os.path.exists(local_scene):
    print('Downloaded {}'.format(local_scene))
    local_scenes.append(local_scene)

In [ ]:
# acolite settings
settings = {## basic input and output settings
            ## local_scenes list is used as returned from the API download function
            'inputfile': local_scenes,
            'output': 'Output/',

            ## set up region of interest centred on lon/lat, with box size in km
            'region_name': region_name,
            'station_lon': station_lon,
            'station_lat': station_lat,
            'station_box_size': 10,
            'station_box_units': 'km',

            ## disable ancillary data retrieval, set to True if EarthData credentials are set
            'ancillary_data': False,

            ## merge scenes if there are more that cover the ROI
            'merge_tiles': True,

            ## compress NetCDF files
            'netcdf_compression': True,

            ## delete L1R output NetCDF
            'l1r_delete_netcdf': True,
            ## don't output L1R RGBs
            'l1r_rgb_keys': [],
            ## delete ACOLITE text outputs
            'delete_acolite_run_text_files': True,
            }

In [ ]:
## run acolite with settings dict
ret = ac.acolite.acolite_run(settings)